1. Import All Libraries

In [ ]:
import pandas as pd #loads and manipulates the Excel dataset
import numpy as np #provides numerical operations and array handling
from sklearn.ensemble import RandomForestClassifier #imports Random Forest model
from sklearn.linear_model import LogisticRegression # imports Logistic Regression model
from sklearn.tree import DecisionTreeClassifier #imports Decision Tree model
from sklearn.model_selection import train_test_split, GridSearchCV #splits data into training and testing sets. GridSearchCV tries many combinations of hyperparameters to find the best ones
from sklearn.preprocessing import LabelEncoder #converts text categories like "Male"/"Female" into numbers 0 and 1
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay #tools to measure how good the model is
)
from imblearn.over_sampling import SMOTE #handles class imbalance by creating synthetic samples of the minority class
from xgboost import XGBClassifier #provides XGBoost model which typically outperforms Random Forest on tabular data
import matplotlib #draws the comparison charts and confusion matrix
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import joblib #saves and loads the trained model as a .pkl file
import warnings
warnings.filterwarnings('ignore') #hides non-critical warning messages to keep output clean

2. Load and Understand the Data : Load the Dataset

In [4]:
print("=" * 60)
print("STEP 1: LOADING DATA")
print("=" * 60)

df = pd.read_excel('../data/E Commerce Dataset.xlsx', sheet_name='E Comm')

STEP 1: LOADING DATA


3. Explore the Data

In [ ]:
print(f"Shape: {df.shape}") #prints (rows, columns) so you know the size of your dataset
print(f"Columns: {df.columns.tolist()}") #prints all column names so you can see what features exist
print(f"\nChurn distribution:\n{df['Churn'].value_counts()}") #shows how many customers are 0 (not churned) and 1 (churned). This tells you if there is class imbalance
print(f"Churn rate: {df['Churn'].mean():.1%}") #shows churn rate as a percentage. If it says 0.165 that means 16.5% of customers churned
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}") #counts missing values per column. We need to handle these before training
print(f"\nSample data:\n{df.head()}") #shows the first 5 rows so you can visually inspect the data

Shape: (5630, 20)
Columns: ['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']

Churn distribution:
Churn
0    4682
1     948
Name: count, dtype: int64
Churn rate: 16.8%

Missing values:
Tenure                         264
WarehouseToHome                251
HourSpendOnApp                 255
OrderAmountHikeFromlastYear    265
CouponUsed                     256
OrderCount                     258
DaySinceLastOrder              307
dtype: int64

Sample data:
   CustomerID  Churn  Tenure PreferredLoginDevice  CityTier  WarehouseToHome  \
0       50001      1     4.0         Mobile Phone         3              6.0   
1       50002      1     NaN                Phone         1              8.0 

4. Preprocessing:- Handle Missing Values (Better Than dropna)

In [ ]:
print("=" * 60)
print("STEP 2: HANDLING MISSING VALUES")
print("=" * 60)

# Fill numeric columns with median
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist() #selects only numeric columns automatically so we don't have to list them manually.
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  Filled {col} with median: {median_val}")

# Fill categorical columns with mode
cat_cols_raw = df.select_dtypes(include=['object']).columns.tolist() #selects only categorical columns automatically. This way if we add new categorical features later, they will be included without code changes.
for col in cat_cols_raw:
    if df[col].isnull().sum() > 0:
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"  Filled {col} with mode: {mode_val}")

print(f"\nMissing values remaining: {df.isnull().sum().sum()}")

STEP 2: HANDLING MISSING VALUES
  Filled Tenure with median: 9.0
  Filled WarehouseToHome with median: 14.0
  Filled HourSpendOnApp with median: 3.0
  Filled OrderAmountHikeFromlastYear with median: 15.0
  Filled CouponUsed with median: 1.0
  Filled OrderCount with median: 2.0
  Filled DaySinceLastOrder with median: 3.0

Missing values remaining: 0


5. Encode Categorical Columns

In [ ]:
print("=" * 60)
print("STEP 3: ENCODING CATEGORICAL COLUMNS")
print("=" * 60)

le = LabelEncoder()

cat_cols_encode = [
    'PreferredLoginDevice',
    'PreferredPaymentMode',
    'Gender',
    'PreferedOrderCat',
    'MaritalStatus',
]

for col in cat_cols_encode:
    original_values = df[col].unique().tolist()
    df[col] = le.fit_transform(df[col]) #fit learns the unique values in the column. transform converts them to integers. fit_transform does both in one step.
    encoded_values = df[col].unique().tolist()
    print(f"  {col}: {original_values} → {encoded_values}")

    #Why these 5 columns specifically: These are the only text (object dtype) columns in the dataset. All other columns are already numeric.

STEP 3: ENCODING CATEGORICAL COLUMNS
  PreferredLoginDevice: ['Mobile Phone', 'Phone', 'Computer'] → [1, 2, 0]
  PreferredPaymentMode: ['Debit Card', 'UPI', 'CC', 'Cash on Delivery', 'E wallet', 'COD', 'Credit Card'] → [4, 6, 0, 2, 5, 1, 3]
  Gender: ['Female', 'Male'] → [0, 1]
  PreferedOrderCat: ['Laptop & Accessory', 'Mobile', 'Mobile Phone', 'Others', 'Fashion', 'Grocery'] → [2, 3, 4, 5, 0, 1]
  MaritalStatus: ['Single', 'Divorced', 'Married'] → [2, 0, 1]


6. Feature Engineering - Create New Features From Existing Ones
What is feature engineering -> Creating new columns by combining or transforming existing ones to give the model stronger signals.
Why the model needs it: The model learns patterns from columns. Raw columns like OrderCount=2 and Tenure=24 individually are weak signals. But order_rate = 2/24 = 0.08 (barely buys despite being a member for 2 years) is a much stronger churn signal.

In [8]:
print("=" * 60)
print("STEP 4: FEATURE ENGINEERING")
print("=" * 60)

STEP 4: FEATURE ENGINEERING


What: Creates a new column that is 1 if the customer has not ordered in more than 10 days, 0 otherwise.
Why 10 days: In e-commerce, 10 days without an order is an early warning sign. You can adjust this threshold.
(df['DaySinceLastOrder'] > 10) — this creates a True/False column.
.astype(int) — converts True to 1 and False to 0 so the model can use it.

In [9]:
# ── Recency feature ───────────────────────────────
df['recency_risk'] = (df['DaySinceLastOrder'] > 10).astype(int)
print("  + recency_risk")

  + recency_risk


What: Orders placed per month of membership.
Why: A customer with 2 orders in 1 month of tenure (rate=2.0) is very different from a customer with 2 orders in 24 months of tenure (rate=0.08). The second is at much higher churn risk.
+ 1 — prevents division by zero if Tenure is 0.

In [10]:
# ── Order frequency ───────────────────────────────
df['order_rate'] = df['OrderCount'] / (df['Tenure'] + 1)
print("  + order_rate")

  + order_rate


What: 1 if the customer has placed 2 or fewer orders total.
Why: Customers who barely buy are the easiest to lose. This gives the model a clear binary signal.

In [11]:
df['low_order_flag'] = (df['OrderCount'] <= 2).astype(int)
print("  + low_order_flag")

  + low_order_flag


What: How much cashback the customer earns per month.
Why: Higher cashback per month means the customer is actively shopping and receiving rewards. Customers with very low cashback per month are not engaged.

In [12]:
# ── Monetary features ─────────────────────────────
df['cashback_per_month'] = df['CashbackAmount'] / (df['Tenure'] + 1)
print("  + cashback_per_month")

  + cashback_per_month


What: Ratio of coupons used per order.
Why: Customers who use coupons regularly are more price-sensitive. If they stop seeing deals they might leave. Also high coupon usage with low orders is a churn pattern.

In [13]:
df['coupon_usage_rate'] = df['CouponUsed'] / (df['OrderCount'] + 1)
print("  + coupon_usage_rate")

  + coupon_usage_rate


What: 1 if the customer spends less than 2 hours on the app.
Why: Low app time means the customer is not browsing or interested. This is a direct disengagement signal.

In [14]:
# ── Engagement features ───────────────────────────
df['low_engagement'] = (df['HourSpendOnApp'] < 2).astype(int)
print("  + low_engagement")

  + low_engagement


What: A single number that summarizes how engaged a customer is overall.
Why weights:

OrderCount gets 0.4 (highest weight) because buying is the most important engagement
HourSpendOnApp gets 0.3 because time on app strongly indicates interest
NumberOfDeviceRegistered and CouponUsed each get 0.15 as supporting signals
Why combine them: Individual signals are noisy. A combined score is more stable and reliable.

In [15]:
df['engagement_score'] = (
    df['HourSpendOnApp']           * 0.3 +
    df['OrderCount']               * 0.4 +
    df['NumberOfDeviceRegistered'] * 0.15 +
    df['CouponUsed']               * 0.15
)
print("  + engagement_score")

  + engagement_score


What: 1 if the customer gave a satisfaction score of 1 or 2, OR if they filed a complaint.
Why: Dissatisfied customers are the most likely to churn. This combines two related signals into one strong feature.
| — means OR in Python. Either condition being true makes the result 1.

In [16]:
# ── Satisfaction risk ─────────────────────────────
df['dissatisfied'] = (
    (df['SatisfactionScore'] <= 2) | (df['Complain'] == 1)
).astype(int)
print("  + dissatisfied")

  + dissatisfied


What: A master risk score combining all the engineered binary flags.
Why weights:

recency_risk 0.30 — not ordering recently is the strongest churn signal
low_order_flag 0.25 — rarely buying is second strongest
low_engagement 0.25 — not using the app is equally important
dissatisfied 0.20 — complaints matter but some complainers still buy
Why this is useful: The model can use this as one clean summary feature alongside all individual features.

In [17]:
# ── Combined churn risk signal ────────────────────
df['churn_risk_score'] = (
    df['recency_risk']    * 0.30 +
    df['low_order_flag']  * 0.25 +
    df['low_engagement']  * 0.25 +
    df['dissatisfied']    * 0.20
)
print("  + churn_risk_score")

print(f"\nTotal features: {len(df.columns) - 2}")

  + churn_risk_score

Total features: 27


6. Prepare for Training - Separate Features and Target

What:
X = all columns except CustomerID and Churn. These are the input features.
y = only the Churn column. This is what we are trying to predict.
Why drop CustomerID: It is just an ID number, not a behavioral feature. If we keep it the model might memorize IDs instead of learning patterns.
Why drop Churn from X: We cannot give the model the answer during training. It must learn to predict Churn from the other columns only.

In [18]:
print("=" * 60)
print("STEP 5: PREPARING X AND y")
print("=" * 60)

X = df.drop(['CustomerID', 'Churn'], axis=1)
y = df['Churn']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nAll feature columns:")
for i, col in enumerate(X.columns):
    print(f"  {i:2d}: {col}")

STEP 5: PREPARING X AND y
X shape: (5630, 27)
y shape: (5630,)

All feature columns:
   0: Tenure
   1: PreferredLoginDevice
   2: CityTier
   3: WarehouseToHome
   4: PreferredPaymentMode
   5: Gender
   6: HourSpendOnApp
   7: NumberOfDeviceRegistered
   8: PreferedOrderCat
   9: SatisfactionScore
  10: MaritalStatus
  11: NumberOfAddress
  12: Complain
  13: OrderAmountHikeFromlastYear
  14: CouponUsed
  15: OrderCount
  16: DaySinceLastOrder
  17: CashbackAmount
  18: recency_risk
  19: order_rate
  20: low_order_flag
  21: cashback_per_month
  22: coupon_usage_rate
  23: low_engagement
  24: engagement_score
  25: dissatisfied
  26: churn_risk_score


7. Train Test Split

In [ ]:
print("=" * 60)
print("STEP 6: TRAIN TEST SPLIT")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, #test_size=0.2 — 20% of rows go to testing, 80% go to training. With 5,630 rows that means about 4,504 for training and 1,126 for testing.
    random_state=42,
    stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set:  {X_test.shape[0]} rows")
print(f"Train churn rate: {y_train.mean():.1%}")
print(f"Test churn rate:  {y_test.mean():.1%}")

STEP 6: TRAIN TEST SPLIT
Training set: 4504 rows
Testing set:  1126 rows
Train churn rate: 16.8%
Test churn rate:  16.9%


8. Apply SMOTE
What is class imbalance: In your dataset about 83% of customers did not churn and only 17% did. This means the model sees far more non-churn examples during training and becomes biased — it learns to just predict "not churn" for everyone and still gets 83% accuracy. But it completely misses actual churners.
What SMOTE does: SMOTE stands for Synthetic Minority Oversampling Technique. It looks at existing churn examples (minority class), finds their nearest neighbors, and creates new synthetic churn examples between them. It does NOT just copy existing rows — it creates new realistic ones.
Why only on training data: We apply SMOTE only to X_train and y_train, never to the test set. The test set must reflect real-world distribution to give honest evaluation results.
random_state=42 — reproducible results.
Result: After SMOTE both classes have equal numbers, so the model learns both equally well.

In [20]:
print("=" * 60)
print("STEP 7: SMOTE — FIXING CLASS IMBALANCE")
print("=" * 60)

print(f"Before SMOTE:")
print(f"  Not Churn (0): {(y_train == 0).sum()}")
print(f"  Churn     (1): {(y_train == 1).sum()}")

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  Not Churn (0): {(y_train_bal == 0).sum()}")
print(f"  Churn     (1): {(y_train_bal == 1).sum()}")

STEP 7: SMOTE — FIXING CLASS IMBALANCE
Before SMOTE:
  Not Churn (0): 3746
  Churn     (1): 758

After SMOTE:
  Not Churn (0): 3746
  Churn     (1): 3746


9. Model Comparison - Define and Train All Models
Why these four models:
i. Logistic Regression — the simplest model. It draws a straight line between churners and non-churners. Fast to train, easy to explain. Used as a baseline. If even Logistic Regression does well, the problem is relatively straightforward.
ii. Decision Tree — asks a series of yes/no questions about features. Example: "Is DaySinceLastOrder > 10? Yes → Is OrderCount < 2? Yes → Predict Churn." Easy to visualize and explain. Can overfit on its own.
iii. Random Forest — builds 100 decision trees on random subsets of data and features, then takes a majority vote. More robust than a single tree because errors from individual trees cancel out. This is our current baseline model.
iv. XGBoost — builds trees sequentially where each new tree focuses on correcting the mistakes of the previous one. Generally the most accurate on tabular data. More complex but worth it for better recall on churners.

max_iter=1000 — Logistic Regression needs more iterations to converge on this dataset. Default is 100 which may not be enough.
n_estimators=100 — Random Forest builds 100 trees.
eval_metric='logloss' — tells XGBoost to use log loss as its internal evaluation metric.
verbosity=0 — suppresses XGBoost training output to keep terminal clean.

In [21]:
print("=" * 60)
print("STEP 8: COMPARING MODELS")
print("=" * 60)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ),
    'XGBoost': XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    ),
}

STEP 8: COMPARING MODELS


10. Train and Evaluate Each Model

Why we care most about Recall for class 1:

Recall = how many actual churners did we correctly identify
Missing a churner (false negative) = we lose a customer we could have saved
False alarm (false positive) = we send an unnecessary retention email, small cost
Therefore missing churners is far more expensive than false alarms

In [ ]:
results = {}

print(f"\n{'Model':<25} {'Accuracy':>10} {'AUC-ROC':>10} "
      f"{'Recall':>10} {'F1':>10} {'Precision':>12}")
print("-" * 75)

for name, model in models.items():

    # Train on SMOTE-balanced data
    model.fit(X_train_bal, y_train_bal) 

    # Predict on REAL test data (no SMOTE)
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] #generates probability scores between 0 and 1 for each customer. [:, 1] takes only the probability of being class 1 (churn). This is what becomes the churn score in Django.

    # Calculate metrics
    report    = classification_report(y_test, preds, output_dict=True) #returns a dictionary of precision, recall, F1-score for each class.
    auc       = roc_auc_score(y_test, proba)
    accuracy  = report['accuracy']
    recall    = report['1']['recall'] #gets recall specifically for class 1 (churners). This is the most important metric for us.
    f1        = report['1']['f1-score']
    precision = report['1']['precision']

    results[name] = {
        'model':     model,
        'auc':       auc,
        'acc':       accuracy,
        'recall':    recall,
        'f1':        f1,
        'precision': precision,
        'preds':     preds,
        'proba':     proba,
    }

    print(f"{name:<25} {accuracy:>10.4f} {auc:>10.4f} "
          f"{recall:>10.4f} {f1:>10.4f} {precision:>12.4f}")


Model                       Accuracy    AUC-ROC     Recall         F1    Precision
---------------------------------------------------------------------------
Logistic Regression           0.8250     0.8408     0.6895     0.5708       0.4870
Decision Tree                 0.9529     0.9339     0.9053     0.8665       0.8309
Random Forest                 0.9707     0.9945     0.9158     0.9134       0.9110
XGBoost                       0.9822     0.9981     0.9421     0.9471       0.9521


11. Hyperparameter Tuning - GridSearchCV on XGBoost

What are hyperparameters: Settings that control how the model learns. They are not learned from data — we set them before training. The default values are often not the best ones.
What GridSearchCV does: Tries every possible combination of the parameters we specify, trains and tests each one using cross-validation, and tells us which combination performed best.

Each parameter explained:

n_estimators — how many trees to build. More trees = more accurate but slower. We try 100, 200, 300.
max_depth — how deep each tree can go. Deeper = learns more complex patterns but may overfit. We try 3, 5, 7.
learning_rate — how much each new tree corrects the previous one. Lower = more trees needed but more accurate. We try 0.01, 0.05, 0.1.
subsample — fraction of training rows used per tree. 0.8 means each tree only sees 80% of rows randomly selected. Prevents overfitting.
colsample_bytree — fraction of features used per tree. 0.8 means each tree only sees 80% of columns randomly selected. Prevents overfitting.

cv=5 — 5-fold cross-validation. Splits training data into 5 parts. Trains on 4, tests on 1, rotates through all 5 combinations, averages the scores. More reliable than a single train-test split.
scoring='roc_auc' — uses AUC-ROC as the measure for selecting the best parameters. We want the model that best separates churners from non-churners.
n_jobs=-1 — uses all CPU cores to run combinations in parallel. Makes GridSearch much faster.

In [23]:
print("=" * 60)
print("STEP 9: HYPERPARAMETER TUNING — XGBOOST")
print("=" * 60)

param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [3, 5, 7],
    'learning_rate':    [0.01, 0.05, 0.1],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

print("Parameters being tested:")
total_combinations = 1
for param, values in param_grid.items():
    print(f"  {param}: {values}")
    total_combinations *= len(values)
print(f"\nTotal combinations: {total_combinations}")
print(f"With 5-fold CV: {total_combinations * 5} model fits")
print("This will take a few minutes...\n")

xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train_bal, y_train_bal)

print(f"\nBest parameters found:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nBest cross-validation AUC: {grid_search.best_score_:.4f}")

STEP 9: HYPERPARAMETER TUNING — XGBOOST
Parameters being tested:
  n_estimators: [100, 200, 300]
  max_depth: [3, 5, 7]
  learning_rate: [0.01, 0.05, 0.1]
  subsample: [0.8, 1.0]
  colsample_bytree: [0.8, 1.0]

Total combinations: 108
With 5-fold CV: 540 model fits
This will take a few minutes...

Fitting 5 folds for each of 108 candidates, totalling 540 fits

Best parameters found:
  colsample_bytree: 0.8
  learning_rate: 0.1
  max_depth: 7
  n_estimators: 300
  subsample: 1.0

Best cross-validation AUC: 0.9950


12. Final Evaluation - Evaluate the Best Model

grid_search.best_estimator_ — retrieves the best model found by GridSearchCV. It is already trained — no need to call .fit() again.
Classification report explained:

Precision — of all customers we predicted will churn, what fraction actually did. High precision = fewer false alarms.
Recall — of all customers who actually churned, what fraction we caught. High recall = fewer missed churners.
F1-score — harmonic mean of precision and recall. Balances both.
Support — how many actual examples of each class exist in the test set.

In [24]:
print("=" * 60)
print("STEP 10: FINAL MODEL EVALUATION")
print("=" * 60)

best_model = grid_search.best_estimator_

y_pred  = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

final_auc = roc_auc_score(y_test, y_proba)

print(f"Final Model: XGBoost with tuned hyperparameters")
print(f"Final AUC-ROC: {final_auc:.4f}")
print(f"\nDetailed Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Not Churn (0)', 'Churn (1)']
))

STEP 10: FINAL MODEL EVALUATION
Final Model: XGBoost with tuned hyperparameters
Final AUC-ROC: 0.9992

Detailed Classification Report:
               precision    recall  f1-score   support

Not Churn (0)       0.99      1.00      1.00       936
    Churn (1)       0.98      0.97      0.98       190

     accuracy                           0.99      1126
    macro avg       0.99      0.98      0.99      1126
 weighted avg       0.99      0.99      0.99      1126



13. Plot Confusion Matrix and Comparison Charts

Confusion Matrix explained:

True Negative — customer was NOT going to churn and we correctly predicted not churn. Good.
False Positive — customer was NOT going to churn but we predicted churn. We send them an unnecessary email. Minor cost.
False Negative — customer WAS going to churn but we missed them. They leave. This is the expensive mistake.
True Positive — customer WAS going to churn and we correctly caught them. We can now send a retention offer. This is the goal.

In [ ]:
print("=" * 60)
print("STEP 11: GENERATING PLOTS")
print("=" * 60)

# ══════════════════════════════════════════════════
# PLOT 1 — Confusion Matrix
# ══════════════════════════════════════════════════
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix values:")
print(f"  True Negative  (correctly said Not Churn): {cm[0][0]}")
print(f"  False Positive (wrongly said Churn):       {cm[0][1]}")
print(f"  False Negative (missed actual Churn):      {cm[1][0]}")
print(f"  True Positive  (correctly said Churn):     {cm[1][1]}")

fig1, ax1 = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not Churn', 'Churn']
)
disp.plot(ax=ax1, colorbar=False, cmap='Blues')
ax1.set_title('Confusion Matrix — XGBoost Tuned', fontsize=13)
plt.tight_layout()
plt.savefig('../plots/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.close(fig1)  # ← close after saving, don't show
print("✅ Saved: confusion_matrix.png")




STEP 11: GENERATING PLOTS
Confusion Matrix values:
  True Negative  (correctly said Not Churn): 932
  False Positive (wrongly said Churn):       4
  False Negative (missed actual Churn):      5
  True Positive  (correctly said Churn):     185
✅ Saved: confusion_matrix.png

Model AUC comparison:
  Logistic Regression : 0.8408
  Decision Tree       : 0.9339
  Random Forest       : 0.9945
  XGBoost             : 0.9981
  XGBoost Tuned       : 0.9992
✅ Saved: model_comparison.png

Checking lengths:
  importances:   27
  feature names: 27

Top 12 feature importances:
  Tenure                             : 0.1813
  churn_risk_score                   : 0.1007
  recency_risk                       : 0.0680
  HourSpendOnApp                     : 0.0637
  cashback_per_month                 : 0.0494
  order_rate                         : 0.0459
  low_order_flag                     : 0.0432
  dissatisfied                       : 0.0392
  Complain                           : 0.0353
  DaySinceLastOrd

What this shows: A bar chart comparing all 5 models by AUC-ROC score. The best model (XGBoost Tuned) is highlighted in blue. 

In [ ]:
# ══════════════════════════════════════════════════
# PLOT 2 — Model Comparison
# ══════════════════════════════════════════════════
model_names = []
model_aucs  = []
model_recalls = []

for name in ['Logistic Regression', 'Decision Tree', 'Random Forest', 'XGBoost']:
    if name in results:
        model_names.append(name)
        model_aucs.append(results[name]['auc'])
        model_recalls.append(results[name]['recall'])

# Add tuned XGBoost separately
model_names.append('XGBoost\nTuned')
model_aucs.append(final_auc)

final_report = classification_report(y_test, y_pred, output_dict=True)
model_recalls.append(final_report['1']['recall'])

print(f"\nModel AUC comparison:")
for name, auc in zip(model_names, model_aucs):
    print(f"  {name.replace(chr(10), ' '):<20}: {auc:.4f}")

colors = ['#cbd5e1', '#94a3b8', '#64748b', '#334155', '#2563eb']

fig2, ax2 = plt.subplots(figsize=(10, 5))
bars = ax2.bar(
    model_names,
    model_aucs,
    color=colors,
    edgecolor='white',
    width=0.5
)
ax2.set_ylim(0.85, 1.02)
ax2.set_title('Model Comparison — AUC-ROC Score', fontsize=14)
ax2.set_ylabel('AUC-ROC Score', fontsize=11)
ax2.set_xlabel('Model', fontsize=11)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

for bar, score in zip(bars, model_aucs):
    ax2.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f'{score:.4f}',
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold'
    )

plt.tight_layout()
plt.savefig('../plots/model_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig2)  # ← close after saving
print("✅ Saved: model_comparison.png")


What feature importance shows: Which columns the model relies on most when making predictions. DaySinceLastOrder and Tenure typically appear at the top. This validates that our feature engineering created useful signals — if churn_risk_score or recency_risk appear high on this chart, our engineering worked.

In [ ]:
# ══════════════════════════════════════════════════
# PLOT 3 — Feature Importance
# ══════════════════════════════════════════════════
importances = best_model.feature_importances_
feature_names = X.columns.tolist()

print(f"\nChecking lengths:")
print(f"  importances:   {len(importances)}")
print(f"  feature names: {len(feature_names)}")

feat_series = pd.Series(
    importances,
    index=feature_names
).sort_values(ascending=True).tail(12)

print(f"\nTop 12 feature importances:")
for feat, score in feat_series.sort_values(ascending=False).items():
    print(f"  {feat:<35}: {score:.4f}")

fig3, ax3 = plt.subplots(figsize=(10, 6))
feat_series.plot(kind='barh', ax=ax3, color='#2563eb')
ax3.set_title('Top 12 Feature Importances — XGBoost Tuned', fontsize=14)
ax3.set_xlabel('Importance Score', fontsize=11)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('../plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.close(fig3)  # ← close after saving
print("✅ Saved: feature_importance.png")


# ══════════════════════════════════════════════════
# COMBINED — All 3 in one file for presentation
# ════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Confusion matrix
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Not Churn', 'Churn']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix\nXGBoost Tuned', fontsize=12)

# Model comparison
bars = axes[1].bar(model_names, model_aucs, color=colors, edgecolor='white', width=0.5)
axes[1].set_ylim(0.85, 1.02)
axes[1].set_title('Model Comparison\nAUC-ROC Score', fontsize=12)
axes[1].set_ylabel('AUC-ROC')
axes[1].tick_params(axis='x', labelsize=8)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)
for bar, score in zip(bars, model_aucs):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.003,
        f'{score:.3f}',
        ha='center', va='bottom', fontsize=8, fontweight='bold'
    )

# Feature importance
feat_series.plot(kind='barh', ax=axes[2], color='#2563eb')
axes[2].set_title('Top 12 Feature Importances\nXGBoost Tuned', fontsize=12)
axes[2].set_xlabel('Importance Score')
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/model_results.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print("✅ Saved: model_results.png (combined)")

print("\n" + "=" * 60)
print("ALL PLOTS SAVED SUCCESSFULLY")
print("Files on your Desktop:")
print("  - confusion_matrix.png")
print("  - model_comparison.png")
print("  - feature_importance.png")
print("  - model_results.png (all three combined)")
print("=" * 60)

14. Save Model - Save Both Files
churn_model.pkl — the complete trained XGBoost model. Contains all the learned decision boundaries. Django loads this file and uses it to score real users.
feature_columns.pkl — a Python list of all column names in the exact order the model was trained on. This is critical. When Django builds a feature dictionary for a real user, it must pass the columns in this exact order. If the order is wrong the model gives incorrect predictions.
joblib.dump — serializes the Python object to a binary file on disk. Much faster than pickle for numpy arrays and sklearn/xgboost models.

In [ ]:
print("=" * 60)
print("STEP 12: SAVING MODEL FILES")
print("=" * 60)

joblib.dump(best_model,         '../models/churn_model.pkl')
joblib.dump(X.columns.tolist(), '../models/feature_columns.pkl')

print("✅ Saved: churn_model.pkl")
print("✅ Saved: feature_columns.pkl")
print(f"\nFeature columns saved ({len(X.columns)}):")
for col in X.columns:
    print(f"  - {col}")
print("\nNext: Copy both files to your Django ml_models/ folder")

STEP 12: SAVING MODEL FILES
✅ Saved: churn_model.pkl
✅ Saved: feature_columns.pkl

Feature columns saved (27):
  - Tenure
  - PreferredLoginDevice
  - CityTier
  - WarehouseToHome
  - PreferredPaymentMode
  - Gender
  - HourSpendOnApp
  - NumberOfDeviceRegistered
  - PreferedOrderCat
  - SatisfactionScore
  - MaritalStatus
  - NumberOfAddress
  - Complain
  - OrderAmountHikeFromlastYear
  - CouponUsed
  - OrderCount
  - DaySinceLastOrder
  - CashbackAmount
  - recency_risk
  - order_rate
  - low_order_flag
  - cashback_per_month
  - coupon_usage_rate
  - low_engagement
  - engagement_score
  - dissatisfied
  - churn_risk_score

Next: Copy both files to your Django ml_models/ folder
